In [ ]:
# 1. Configure the processing environment
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, binary_opening, binary_closing, binary_dilation
from skimage import io, filters, morphology, exposure
from sklearn.mixture import GaussianMixture

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "catheter").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "catheter"
OUT = ROOT / "outputs"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
REFERENCE_FRAME = 201
FRAMES = list(range(211, 292, 10))
REPRESENTATIVE = 251
GAUSSIAN_SIGMA = 1.0
PERCENTILES = (1.0, 99.0)
HP_CUTOFF = 16.0
FIXED_THRESHOLD = 0.32
MORPH_RADIUS = 1
GMM_RANDOM_STATE = 42
EPS = 1e-9
print("RECONSTRUCTED IMPLEMENTATION — the original main.ipynb was empty.")
print("Parameters are reproducible estimates, NOT verified originals.")
print(f"Project root: {ROOT}; evaluation frames: {FRAMES}")

In [ ]:
# 2. Reusable processing and evaluation functions
def as_gray(path):
    img = np.asarray(io.imread(str(path)))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=-1)
    return img.astype(np.float32)

def norm_percentile(img, lo=PERCENTILES[0], hi=PERCENTILES[1]):
    a, b = np.percentile(img, (lo, hi))
    return np.clip((img - a) / max(float(b - a), EPS), 0, 1).astype(np.float32)

def load_frame(number):
    return as_gray(DATA / f"frame_{number}.png")

def annotation_tool_mask(path):
    a = as_gray(path)
    # Annotations should be sparse; interpret the minority class as tool.
    # If the binary polarity is ambiguous, fail rather than silently invert it.
    vals = np.unique(a)
    if vals.size > 32:
        raise ValueError(f"Unexpected annotation grayscale levels in {path.name}: {vals.size}")
    middle = (float(vals.min()) + float(vals.max())) / 2
    bright = a > middle
    frac = float(bright.mean())
    if frac < 0.20:
        return bright
    if frac > 0.80:
        return ~bright
    raise ValueError(f"Ambiguous annotation polarity ({frac:.1%} bright) in {path.name}")

def load_ground_truth(number):
    masks = [annotation_tool_mask(DATA / f"{number}_{kind}.tiff")
             for kind in ("GuideWire", "MicroCath")]
    assert masks[0].shape == masks[1].shape
    return np.logical_or.reduce(masks)

def preprocess(img):
    return norm_percentile(gaussian_filter(img, GAUSSIAN_SIGMA))

def gaussian_highpass(signed, cutoff=HP_CUTOFF):
    h, w = signed.shape
    y, x = np.ogrid[:h, :w]
    r2 = (y - h // 2) ** 2 + (x - w // 2) ** 2
    H = 1.0 - np.exp(-r2 / (2.0 * cutoff ** 2))
    F = np.fft.fftshift(np.fft.fft2(signed))
    filtered = np.real(np.fft.ifft2(np.fft.ifftshift(F * H)))
    return filtered, np.log1p(np.abs(F)), H

def construct_feature(img, background):
    processed = preprocess(img)
    signed = background - processed
    spectral, fft_mag, transfer = gaussian_highpass(signed)
    positive = np.maximum(spectral, 0)
    feature = norm_percentile(gaussian_filter(positive, 0.7))
    return dict(processed=processed, signed=signed, spectral=spectral,
                fft_magnitude=fft_mag, transfer=transfer, feature=feature)

def finalize_tool(raw):
    raw = np.asarray(raw, dtype=bool)
    return binary_closing(raw, structure=morphology.disk(MORPH_RADIUS))

def segment(feature, strategy):
    if strategy == "Fixed Threshold":
        raw = feature > FIXED_THRESHOLD
    elif strategy == "Otsu":
        raw = feature > filters.threshold_otsu(feature)
    elif strategy == "EM / GMM":
        values = feature.reshape(-1, 1)
        # Fit on representative subsample to keep the per-frame notebook usable.
        rng = np.random.default_rng(GMM_RANDOM_STATE)
        sample = values[rng.choice(len(values), size=min(30000, len(values)), replace=False)]
        model = GaussianMixture(n_components=2, random_state=GMM_RANDOM_STATE,
                                reg_covar=1e-5, max_iter=150).fit(sample)
        raw = model.predict(values).reshape(feature.shape) == np.argmax(model.means_)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    tool = finalize_tool(raw)
    mask_01 = (~tool).astype(np.uint8)  # mandated convention: 0=tool, 1=background
    return tool, mask_01

STRATEGIES = ("Fixed Threshold", "Otsu", "EM / GMM")

def metrics(gt_tool, pred_tool):
    gt = np.asarray(gt_tool, bool)
    pred = np.asarray(pred_tool, bool)
    assert gt.shape == pred.shape
    # Historical CSV measures pixel error on 0/255 masks.
    diff = (gt.astype(np.float32) - pred.astype(np.float32)) * 255.0
    sad = float(np.abs(diff).mean())
    mse = float(np.square(diff).mean())
    psnr = float(20 * np.log10(255.0 / np.sqrt(max(mse, EPS))))
    overlap = int(np.logical_and(gt, pred).sum())
    total = int(gt.sum() + pred.sum())
    union = int(np.logical_or(gt, pred).sum())
    return dict(SAD=sad, MSE=mse, PSNR=psnr,
                Dice=2 * overlap / total if total else 1.0,
                IoU=overlap / union if union else 1.0)

def semantic_overlay(img, gt, pred):
    rgb = np.repeat(norm_percentile(img)[..., None], 3, axis=2)
    both = gt & pred
    missed = gt & ~pred
    false_positive = pred & ~gt
    rgb[both] = [0.10, 0.85, 0.25]        # agreement / green
    rgb[missed] = [0.95, 0.20, 0.20]      # missed / red
    rgb[false_positive] = [0.20, 0.55, 1.0]  # false positive / blue
    return rgb

def guidance_overlay(img, pred, alpha=0.65):
    rgb = np.repeat(norm_percentile(img)[..., None], 3, axis=2)
    rgb[pred] = (1 - alpha) * rgb[pred] + alpha * np.array([0.1, 1.0, 0.2])
    return rgb

def save_figure(name):
    plt.savefig(FIG / name, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# 3. Validate dataset and annotation geometry
inventory = []
for n in [REFERENCE_FRAME] + FRAMES:
    p = DATA / f"frame_{n}.png"
    assert p.is_file(), f"Missing frame: {p}"
    frame = load_frame(n)
    record = dict(Frame=n, Height=frame.shape[0], Width=frame.shape[1])
    if n != REFERENCE_FRAME:
        for kind in ("GuideWire", "MicroCath"):
            path = DATA / f"{n}_{kind}.tiff"
            assert path.is_file(), f"Missing annotation: {path}"
            annotation = annotation_tool_mask(path)
            assert annotation.shape == frame.shape, f"Shape mismatch: {path}"
            record[kind + "_tool_pixels"] = int(annotation.sum())
    inventory.append(record)
inventory_df = pd.DataFrame(inventory)
assert inventory_df[["Height", "Width"]].nunique().max() == 1
display(inventory_df)

In [ ]:
# 4. Fixed background reference, frame 201
background_raw = load_frame(REFERENCE_FRAME)
background_reference = preprocess(background_raw)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(background_raw, cmap="gray"); ax[0].set_title("Frame 201 — raw reference")
ax[1].imshow(background_reference, cmap="gray", vmin=0, vmax=1); ax[1].set_title("Preprocessed fixed background")
for a in ax: a.axis("off")
save_figure("01_background_reference.png")

In [ ]:
# 5. Representative feature-processing chain, frame 251
representative_image = load_frame(REPRESENTATIVE)
representative_pipeline = construct_feature(representative_image, background_reference)
views = [
    ("Input", representative_image),
    ("Gaussian + percentile", representative_pipeline["processed"]),
    ("Signed background residual", representative_pipeline["signed"]),
    ("Log FFT magnitude", representative_pipeline["fft_magnitude"]),
    ("Gaussian high-pass transfer", representative_pipeline["transfer"]),
    ("Final feature", representative_pipeline["feature"]),
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (label, data) in zip(axes.flat, views):
    ax.imshow(data, cmap="gray"); ax.set_title(label); ax.axis("off")
plt.tight_layout()
save_figure("02_intermediate_pipeline.png")

In [ ]:
# 6. Apply candidate segmentations with the same upstream feature
representative_gt = load_ground_truth(REPRESENTATIVE)
representative_predictions = {
    name: segment(representative_pipeline["feature"], name)[0]
    for name in STRATEGIES
}
for name, mask in representative_predictions.items():
    print(f"{name}: {mask.sum()} predicted tool pixels; metrics: {metrics(representative_gt, mask)}")

In [ ]:
# 7. Qualitative comparison on frame 251
fig, axes = plt.subplots(3, 2, figsize=(11, 14))
for row, name in enumerate(STRATEGIES):
    pred = representative_predictions[name]
    axes[row, 0].imshow(pred, cmap="gray"); axes[row, 0].set_title(name + " — tool mask")
    axes[row, 1].imshow(semantic_overlay(representative_image, representative_gt, pred))
    axes[row, 1].set_title(name + " — green TP / red FN / blue FP")
for ax in axes.flat: ax.axis("off")
plt.tight_layout()
save_figure("03_strategy_comparison_representative.png")

In [ ]:
# 8. Evaluate every strategy on all annotated frames
records = []
cache = {}
for frame_number in FRAMES:
    image = load_frame(frame_number)
    gt_tool = load_ground_truth(frame_number)
    pipeline = construct_feature(image, background_reference)
    cache[frame_number] = dict(image=image, gt=gt_tool, feature=pipeline["feature"], predictions={})
    for name in STRATEGIES:
        predicted_tool, predicted_mask_01 = segment(pipeline["feature"], name)
        cache[frame_number]["predictions"][name] = predicted_tool
        records.append(dict(Frame=frame_number, Strategy=name, **metrics(gt_tool, predicted_tool)))
results_df = pd.DataFrame.from_records(records)
assert len(results_df) == len(FRAMES) * len(STRATEGIES)
display(results_df)

In [ ]:
# 9. Aggregate results and select minimum mean MSE
summary_df = results_df.groupby("Strategy").agg(
    **{f"{metric}_{stat.capitalize()}": (metric, stat)
       for metric in ("SAD", "MSE", "PSNR", "Dice", "IoU")
       for stat in ("mean", "std")}
).reset_index().sort_values("MSE_Mean").reset_index(drop=True)
best_strategy = str(summary_df.iloc[0]["Strategy"])
print("Selected by lowest average MSE:", best_strategy)
display(summary_df)
# Compare with the archived run; differences are expected because original parameters were lost.
archive_path = OUT / "background_subtraction_summary.csv"
if archive_path.exists():
    archived = pd.read_csv(archive_path)
    print("Archived (original run) vs reconstructed values:")
    display(archived[["Strategy", "MSE_Mean", "PSNR_Mean", "Dice_Mean"]])
    display(summary_df[["Strategy", "MSE_Mean", "PSNR_Mean", "Dice_Mean"]])

In [ ]:
# 10. Temporal variation of pixel metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ("SAD", "MSE", "PSNR")):
    for name in STRATEGIES:
        part = results_df.loc[results_df.Strategy == name].sort_values("Frame")
        ax.plot(part.Frame, part[metric], marker="o", label=name)
    ax.set_title(metric); ax.set_xlabel("Frame"); ax.grid(alpha=.25)
axes[-1].legend(loc="best", fontsize=8)
plt.tight_layout()
save_figure("04_temporal_metric_curves.png")

In [ ]:
# 11. Strategy-level quantitative comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ("MSE", "PSNR", "Dice")):
    ax.bar(summary_df.Strategy, summary_df[f"{metric}_Mean"])
    ax.set_title(f"Mean {metric}"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
save_figure("05_strategy_summary.png")

In [ ]:
# 12. Assemble the reusable final pipeline
def final_pipeline(frame_number, strategy_name=None):
    strategy_name = best_strategy if strategy_name is None else strategy_name
    image = load_frame(frame_number)
    values = construct_feature(image, background_reference)
    predicted_tool, mask_01 = segment(values["feature"], strategy_name)
    gt = load_ground_truth(frame_number)
    return dict(image=image, feature=values["feature"], predicted_tool=predicted_tool,
                mask_01=mask_01, ground_truth_tool=gt, strategy=strategy_name)
check = final_pipeline(REPRESENTATIVE)
assert set(np.unique(check["mask_01"])) <= {0, 1}
print("Pipeline OK", check["strategy"], check["image"].shape,
      "tool pixels:", check["predicted_tool"].sum())

In [ ]:
# 13. Final tool-guidance gallery over the evaluation sequence
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, frame_number in zip(axes.flat, FRAMES):
    item = final_pipeline(frame_number)
    ax.imshow(guidance_overlay(item["image"], item["predicted_tool"]))
    ax.set_title(f"Frame {frame_number} — {best_strategy}")
    ax.axis("off")
plt.tight_layout()
save_figure("06_final_guidance_gallery.png")

In [ ]:
# 14. Qualitative validation of early / middle / late frames
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for row, frame_number in enumerate((211, 251, 291)):
    item = final_pipeline(frame_number)
    axes[row, 0].imshow(item["predicted_tool"], cmap="gray")
    axes[row, 1].imshow(item["ground_truth_tool"], cmap="gray")
    axes[row, 2].imshow(semantic_overlay(item["image"], item["ground_truth_tool"], item["predicted_tool"]))
    for col, title in enumerate(("Predicted tool", "Manual annotation", "TP green / FN red / FP blue")):
        axes[row, col].set_title(f"{frame_number}: {title}")
        axes[row, col].axis("off")
plt.tight_layout()
save_figure("07_final_validation.png")

In [ ]:
# 15. Export reconstructed results separately; never overwrite historical evidence
reconstructed_metrics = OUT / "background_subtraction_metrics_reconstructed.csv"
reconstructed_summary = OUT / "background_subtraction_summary_reconstructed.csv"
results_df.to_csv(reconstructed_metrics, index=False)
summary_df.to_csv(reconstructed_summary, index=False)
expected = [f"{i:02d}_{name}.png" for i, name in [
    (1, "background_reference"), (2, "intermediate_pipeline"),
    (3, "strategy_comparison_representative"), (4, "temporal_metric_curves"),
    (5, "strategy_summary"), (6, "final_guidance_gallery"), (7, "final_validation")
]]
assert all((FIG / name).is_file() for name in expected), "Missing figures"
print("Reconstructed outputs saved separately; original published CSVs preserved.")
for item in [reconstructed_metrics, reconstructed_summary, *(FIG / n for n in expected)]:
    print(item.relative_to(ROOT), item.stat().st_size, "bytes")